# Creación del catálogo maestro

In [9]:
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

import astropy.units as u
from astropy.coordinates import SkyCoord

In [3]:
ruta_manifiesto = Path(
    "../data/interim/manifiesto_curvas_validas.csv"
)

catalogo = pd.read_csv(
    ruta_manifiesto,
    dtype={
        "proyecto": "string",
        "region": "string",
        "star_id": "string",
        "ruta_curva": "string",
        "subtipo": "string",
        "archivo_periodos": "string",
    },
)

duplicados = catalogo.duplicated(
    subset=["proyecto", "star_id"]
).sum()

print(f"Archivo encontrado: {ruta_manifiesto.exists()}")
print(f"Filas cargadas: {len(catalogo):,}")
print(f"Columnas: {len(catalogo.columns)}")
print(f"Claves proyecto–star_id duplicadas: {duplicados:,}\n")

print("Columnas disponibles:")
for columna in catalogo.columns:
    print(f"  {columna}")

Archivo encontrado: True
Filas cargadas: 104,008
Columnas: 14
Claves proyecto–star_id duplicadas: 0

Columnas disponibles:
  proyecto
  region
  star_id
  n_obs
  ruta_curva
  total_filas
  n_obs_validas
  formato_invalido
  valores_no_numericos
  error_no_positivo
  subtipo
  periodo_principal
  periodo_secundario
  archivo_periodos


In [4]:
ruta_ident = Path("../data/raw/ident")

archivos_ident = sorted(ruta_ident.glob("ident-*.dat"))

registros_ident = []

for archivo in archivos_ident:
    partes = archivo.stem.split("-")
    region = partes[1]
    proyecto = f"{partes[2]}-{partes[3]}"

    with archivo.open("r", encoding="utf-8") as contenido:
        for linea in contenido:
            if not linea.strip() or linea.lstrip().startswith("#"):
                continue

            campos = linea.split()

            registros_ident.append({
                "proyecto": proyecto,
                "region": region,
                "star_id": campos[0],
                "ra": campos[2],
                "dec": campos[3],
                "archivo_ident": archivo.name,
            })

coordenadas_ident = pd.DataFrame(registros_ident)

catalogo_maestro = catalogo.merge(
    coordenadas_ident,
    on=["proyecto", "region", "star_id"],
    how="left",
    validate="one_to_one",
    indicator=True,
)

print(f"Archivos ident leídos: {len(archivos_ident)}")
print(f"Registros en los ident: {len(coordenadas_ident):,}")
print(f"Filas del catálogo maestro: {len(catalogo_maestro):,}\n")

print("Resultado del cruce:")
print(catalogo_maestro["_merge"].value_counts().to_string())

print(
    "\nFilas sin coordenadas: "
    f"{catalogo_maestro[['ra', 'dec']].isna().any(axis=1).sum():,}"
)

Archivos ident leídos: 7
Registros en los ident: 172,690
Filas del catálogo maestro: 104,008

Resultado del cruce:
_merge
both          104008
left_only          0
right_only         0

Filas sin coordenadas: 0


In [10]:
lineas_ogle_iii_atipicas = []
distribucion_ogle_iii = Counter()

for archivo in archivos_ident:
    if "OGLE-III" not in archivo.stem:
        continue

    with archivo.open("r", encoding="utf-8") as contenido:
        for numero_linea, linea in enumerate(contenido, start=1):
            if not linea.strip() or linea.lstrip().startswith("#"):
                continue

            campos = linea.split()
            distribucion_ogle_iii[len(campos)] += 1

            if len(campos) < 6:
                lineas_ogle_iii_atipicas.append({
                    "archivo": archivo.name,
                    "linea": numero_linea,
                    "n_columnas": len(campos),
                    "contenido": repr(linea.rstrip()),
                })

print("Distribución de columnas en OGLE-III:")

for cantidad_columnas, cantidad_filas in sorted(
    distribucion_ogle_iii.items()
):
    print(
        f"  {cantidad_columnas} columnas: "
        f"{cantidad_filas:,} filas"
    )

print(
    "\nLíneas con menos de 6 columnas: "
    f"{len(lineas_ogle_iii_atipicas):,}\n"
)

for registro in lineas_ogle_iii_atipicas[:30]:
    print(registro)

Distribución de columnas en OGLE-III:
  5 columnas: 292 filas
  6 columnas: 25,195 filas
  7 columnas: 14,821 filas
  8 columnas: 3,641 filas
  9 columnas: 167 filas
  10 columnas: 100 filas
  11 columnas: 1 filas

Líneas con menos de 6 columnas: 292

{'archivo': 'ident-BLG-OGLE-III.dat', 'linea': 122, 'n_columnas': 5, 'contenido': "'OGLE-BLG-RRLYR-00122  BLG365.3        RRab  17:10:53.12 -31:54:45.9'"}
{'archivo': 'ident-BLG-OGLE-III.dat', 'linea': 123, 'n_columnas': 5, 'contenido': "'OGLE-BLG-RRLYR-00123  BLG365.2        RRab  17:10:53.42 -32:06:20.4'"}
{'archivo': 'ident-BLG-OGLE-III.dat', 'linea': 266, 'n_columnas': 5, 'contenido': "'OGLE-BLG-RRLYR-00266  BLG364.3        RRc   17:15:02.19 -31:50:59.9'"}
{'archivo': 'ident-BLG-OGLE-III.dat', 'linea': 273, 'n_columnas': 5, 'contenido': "'OGLE-BLG-RRLYR-00273  BLG358.6        RRc   17:17:39.50 -28:56:33.7'"}
{'archivo': 'ident-BLG-OGLE-III.dat', 'linea': 306, 'n_columnas': 5, 'contenido': "'OGLE-BLG-RRLYR-00306  BLG358.3        RRab  

In [11]:
subtipos_validos = {"RRab", "RRc", "RRd", "RRe", "aRRd"}

registros_ident = []
incidencias_ident = []

for archivo in archivos_ident:
    partes = archivo.stem.split("-")
    region = partes[1]
    proyecto = f"{partes[2]}-{partes[3]}"

    with archivo.open("r", encoding="utf-8") as contenido:
        for numero_linea, linea in enumerate(contenido, start=1):
            if not linea.strip() or linea.lstrip().startswith("#"):
                continue

            campos = linea.split()

            try:
                subtipo_ident = next(
                    campo
                    for campo in campos
                    if campo in subtipos_validos
                )

                ra = next(
                    campo
                    for campo in campos
                    if campo.count(":") == 2
                    and not campo.startswith(("+", "-"))
                )

                dec = next(
                    campo
                    for campo in campos
                    if campo.count(":") == 2
                    and campo.startswith(("+", "-"))
                )

            except StopIteration:
                incidencias_ident.append({
                    "archivo": archivo.name,
                    "linea": numero_linea,
                    "contenido": repr(linea.rstrip()),
                })
                continue

            registros_ident.append({
                "proyecto": proyecto,
                "region": region,
                "star_id": campos[0],
                "subtipo_ident": subtipo_ident,
                "ra": ra,
                "dec": dec,
                "archivo_ident": archivo.name,
            })

coordenadas_ident = pd.DataFrame(registros_ident)

catalogo_maestro = catalogo.merge(
    coordenadas_ident,
    on=["proyecto", "region", "star_id"],
    how="left",
    validate="one_to_one",
    indicator=True,
)

print(f"Registros leídos correctamente: {len(coordenadas_ident):,}")
print(f"Incidencias de lectura: {len(incidencias_ident):,}")
print(f"Filas del catálogo maestro: {len(catalogo_maestro):,}")
print(
    "Filas sin coordenadas: "
    f"{catalogo_maestro[['ra', 'dec']].isna().any(axis=1).sum():,}"
)

Registros leídos correctamente: 172,690
Incidencias de lectura: 0
Filas del catálogo maestro: 104,008
Filas sin coordenadas: 0


In [12]:
ogle_iii = (
    catalogo_maestro[
        catalogo_maestro["proyecto"] == "OGLE-III"
    ][["star_id", "region", "ra", "dec"]]
    .rename(columns={
        "region": "region_iii",
        "ra": "ra_iii",
        "dec": "dec_iii",
    })
)

ogle_iv = (
    catalogo_maestro[
        catalogo_maestro["proyecto"] == "OGLE-IV"
    ][["star_id", "region", "ra", "dec"]]
    .rename(columns={
        "region": "region_iv",
        "ra": "ra_iv",
        "dec": "dec_iv",
    })
)

pares_ogle = ogle_iii.merge(
    ogle_iv,
    on="star_id",
    how="inner",
    validate="one_to_one",
)

coordenadas_iii = SkyCoord(
    ra=pares_ogle["ra_iii"].to_numpy(),
    dec=pares_ogle["dec_iii"].to_numpy(),
    unit=(u.hourangle, u.deg),
)

coordenadas_iv = SkyCoord(
    ra=pares_ogle["ra_iv"].to_numpy(),
    dec=pares_ogle["dec_iv"].to_numpy(),
    unit=(u.hourangle, u.deg),
)

pares_ogle["separacion_arcsec"] = (
    coordenadas_iii.separation(coordenadas_iv).arcsec
)

regiones_distintas = (
    pares_ogle["region_iii"] != pares_ogle["region_iv"]
).sum()

n_estrellas_fisicas = len(catalogo_maestro) - len(pares_ogle)

print(f"Pares OGLE-III/OGLE-IV: {len(pares_ogle):,}")
print(f"Pares con regiones distintas: {regiones_distintas:,}")
print(
    "Separación menor o igual a 1″: "
    f"{(pares_ogle['separacion_arcsec'] <= 1).sum():,}"
)
print(
    "Separación entre 1″ y 2″: "
    f"{(
        (pares_ogle['separacion_arcsec'] > 1)
        & (pares_ogle['separacion_arcsec'] <= 2)
    ).sum():,}"
)
print(
    "Separación mayor que 2″: "
    f"{(pares_ogle['separacion_arcsec'] > 2).sum():,}"
)
print(
    "Separación máxima: "
    f"{pares_ogle['separacion_arcsec'].max():.4f}″"
)
print(f"\nEstrellas físicas únicas: {n_estrellas_fisicas:,}")

Pares OGLE-III/OGLE-IV: 33,739
Pares con regiones distintas: 0
Separación menor o igual a 1″: 33,717
Separación entre 1″ y 2″: 22
Separación mayor que 2″: 0
Separación máxima: 1.9616″

Estrellas físicas únicas: 70,269


In [13]:
ids_con_dos_fases = set(pares_ogle["star_id"])

catalogo_maestro = catalogo_maestro.drop(
    columns="_merge"
).copy()

catalogo_maestro["observacion_id"] = (
    catalogo_maestro["proyecto"]
    .str.replace("-", "", regex=False)
    + "_"
    + catalogo_maestro["star_id"]
)

catalogo_maestro["star_id_base"] = (
    catalogo_maestro["star_id"]
)

catalogo_maestro["dos_observaciones"] = (
    catalogo_maestro["star_id"].isin(ids_con_dos_fases)
)

distribucion_por_estrella = (
    catalogo_maestro
    .groupby("star_id_base")
    .size()
    .value_counts()
    .sort_index()
)

print(f"Filas totales: {len(catalogo_maestro):,}")
print(
    "observacion_id únicos: "
    f"{catalogo_maestro['observacion_id'].nunique():,}"
)
print(
    "star_id_base únicos: "
    f"{catalogo_maestro['star_id_base'].nunique():,}\n"
)

print("Cantidad de observaciones por estrella física:")
print(distribucion_por_estrella.to_string())

print(
    "\nFilas marcadas con dos observaciones: "
    f"{catalogo_maestro['dos_observaciones'].sum():,}"
)

Filas totales: 104,008
observacion_id únicos: 104,008
star_id_base únicos: 70,269

Cantidad de observaciones por estrella física:
1    36530
2    33739

Filas marcadas con dos observaciones: 67,478


In [14]:
catalogo_maestro["subtipo_coincidente"] = (
    catalogo_maestro["subtipo"]
    == catalogo_maestro["subtipo_ident"]
)

discordancias_fuente = catalogo_maestro[
    ~catalogo_maestro["subtipo_coincidente"]
].copy()

print(
    "Observaciones con subtipo coincidente: "
    f"{catalogo_maestro['subtipo_coincidente'].sum():,}"
)
print(
    "Observaciones con subtipo diferente: "
    f"{len(discordancias_fuente):,}"
)

if not discordancias_fuente.empty:
    print("\nComparación de subtipos diferentes:")
    print(
        pd.crosstab(
            discordancias_fuente["subtipo"],
            discordancias_fuente["subtipo_ident"],
        ).to_string()
    )

Observaciones con subtipo coincidente: 104,008
Observaciones con subtipo diferente: 0


In [15]:
subtipos_por_fase = (
    catalogo_maestro
    .pivot(
        index="star_id_base",
        columns="proyecto",
        values="subtipo",
    )
)

pares_con_subtipo = (
    subtipos_por_fase
    .dropna(subset=["OGLE-III", "OGLE-IV"])
    .copy()
)

pares_con_subtipo["subtipos_discordantes"] = (
    pares_con_subtipo["OGLE-III"]
    != pares_con_subtipo["OGLE-IV"]
)

discordancias_subtipo = pares_con_subtipo[
    pares_con_subtipo["subtipos_discordantes"]
].copy()

print(f"Pares comparados: {len(pares_con_subtipo):,}")
print(
    "Pares con el mismo subtipo: "
    f"{(~pares_con_subtipo['subtipos_discordantes']).sum():,}"
)
print(
    "Pares con subtipos discordantes: "
    f"{len(discordancias_subtipo):,}\n"
)

print("Matriz de cambios OGLE-III → OGLE-IV:")

print(
    pd.crosstab(
        discordancias_subtipo["OGLE-III"],
        discordancias_subtipo["OGLE-IV"],
        margins=True,
    ).to_string()
)

Pares comparados: 33,739
Pares con el mismo subtipo: 32,509
Pares con subtipos discordantes: 1,230

Matriz de cambios OGLE-III → OGLE-IV:
OGLE-IV   RRab   RRc  RRd  aRRd   All
OGLE-III                             
RRab         0     3    5    11    19
RRc          4     0   16     4    24
RRd          8     4    0     0    12
RRe          0  1175    0     0  1175
All         12  1182   21    15  1230


In [16]:
mapa_subtipo_iii = subtipos_por_fase["OGLE-III"]
mapa_subtipo_iv = subtipos_por_fase["OGLE-IV"]

mapa_separacion = (
    pares_ogle
    .set_index("star_id")["separacion_arcsec"]
)

catalogo_maestro["subtipo_ogleiii"] = (
    catalogo_maestro["star_id_base"]
    .map(mapa_subtipo_iii)
)

catalogo_maestro["subtipo_ogleiv"] = (
    catalogo_maestro["star_id_base"]
    .map(mapa_subtipo_iv)
)

catalogo_maestro["subtipos_discordantes"] = (
    catalogo_maestro["subtipo_ogleiii"].notna()
    & catalogo_maestro["subtipo_ogleiv"].notna()
    & (
        catalogo_maestro["subtipo_ogleiii"]
        != catalogo_maestro["subtipo_ogleiv"]
    )
)

catalogo_maestro["transicion"] = pd.Series(
    pd.NA,
    index=catalogo_maestro.index,
    dtype="string",
)

mascara_discordante = catalogo_maestro["subtipos_discordantes"]

catalogo_maestro.loc[
    mascara_discordante,
    "transicion",
] = (
    catalogo_maestro.loc[
        mascara_discordante,
        "subtipo_ogleiii",
    ]
    + " → "
    + catalogo_maestro.loc[
        mascara_discordante,
        "subtipo_ogleiv",
    ]
)

catalogo_maestro["separacion_fases_arcsec"] = (
    catalogo_maestro["star_id_base"]
    .map(mapa_separacion)
)

catalogo_maestro["grupo_analisis"] = np.select(
    [
        catalogo_maestro["subtipos_discordantes"],
        catalogo_maestro["subtipo"].isin(["RRe", "aRRd"]),
    ],
    [
        "discordante",
        "minoritaria",
    ],
    default="principal",
)

print("Filas por grupo de análisis:")

print(
    catalogo_maestro["grupo_analisis"]
    .value_counts()
    .to_string()
)

print("\nEstrellas físicas únicas por grupo:")

print(
    catalogo_maestro
    .drop_duplicates("star_id_base")
    ["grupo_analisis"]
    .value_counts()
    .to_string()
)

print(
    "\nEstrellas físicas discordantes: "
    f"{catalogo_maestro.loc[
        catalogo_maestro['subtipos_discordantes'],
        'star_id_base'
    ].nunique():,}"
)

Filas por grupo de análisis:
grupo_analisis
principal      101364
discordante      2460
minoritaria       184

Estrellas físicas únicas por grupo:
grupo_analisis
principal      68855
discordante     1230
minoritaria      184

Estrellas físicas discordantes: 1,230


In [17]:
print(f"Total de columnas actuales: {len(catalogo_maestro.columns)}\n")

for numero, columna in enumerate(
    catalogo_maestro.columns,
    start=1,
):
    print(
        f"{numero:02d}. "
        f"{columna} "
        f"({catalogo_maestro[columna].dtype})"
    )

Total de columnas actuales: 28

01. proyecto (object)
02. region (object)
03. star_id (object)
04. n_obs (int64)
05. ruta_curva (string)
06. total_filas (int64)
07. n_obs_validas (int64)
08. formato_invalido (int64)
09. valores_no_numericos (int64)
10. error_no_positivo (int64)
11. subtipo (string)
12. periodo_principal (float64)
13. periodo_secundario (float64)
14. archivo_periodos (string)
15. subtipo_ident (str)
16. ra (str)
17. dec (str)
18. archivo_ident (str)
19. observacion_id (object)
20. star_id_base (object)
21. dos_observaciones (bool)
22. subtipo_coincidente (bool[pyarrow])
23. subtipo_ogleiii (string)
24. subtipo_ogleiv (string)
25. subtipos_discordantes (bool[pyarrow])
26. transicion (string)
27. separacion_fases_arcsec (float64)
28. grupo_analisis (str)


In [18]:
catalogo_maestro = catalogo_maestro.rename(
    columns={
        "proyecto": "proyecto_origen",
        "region": "region_origen",
        "subtipo": "subtipo_rrlyrae",
        "periodo_principal": "periodo_catalogo",
    }
)

catalogo_maestro["archivo_origen"] = (
    catalogo_maestro["ruta_curva"]
    .map(lambda ruta: Path(ruta).name)
)

catalogo_maestro["region_catalogo"] = (
    catalogo_maestro["star_id"]
    .str.split("-")
    .str[1]
)

catalogo_maestro["tipo_catalogo"] = (
    catalogo_maestro["subtipo_ident"]
)

catalogo_maestro["seleccion_inicial"] = True

assert (
    catalogo_maestro["region_origen"]
    == catalogo_maestro["region_catalogo"]
).all()

assert (
    catalogo_maestro["subtipo_rrlyrae"]
    == catalogo_maestro["tipo_catalogo"]
).all()

columnas_ordenadas = [
    "observacion_id",
    "star_id_base",
    "star_id",
    "proyecto_origen",
    "region_origen",
    "region_catalogo",
    "subtipo_rrlyrae",
    "subtipo_ogleiii",
    "subtipo_ogleiv",
    "subtipos_discordantes",
    "transicion",
    "grupo_analisis",
    "dos_observaciones",
    "periodo_catalogo",
    "periodo_secundario",
    "n_obs",
    "n_obs_validas",
    "ra",
    "dec",
    "separacion_fases_arcsec",
    "archivo_origen",
    "ruta_curva",
    "archivo_periodos",
    "archivo_ident",
    "tipo_catalogo",
    "seleccion_inicial",
    "total_filas",
    "formato_invalido",
    "valores_no_numericos",
    "error_no_positivo",
    "subtipo_ident",
    "subtipo_coincidente",
]

catalogo_maestro = catalogo_maestro[columnas_ordenadas]

print(f"Filas: {len(catalogo_maestro):,}")
print(f"Columnas finales actuales: {len(catalogo_maestro.columns)}\n")

for numero, columna in enumerate(
    catalogo_maestro.columns,
    start=1,
):
    print(f"{numero:02d}. {columna}")

Filas: 104,008
Columnas finales actuales: 32

01. observacion_id
02. star_id_base
03. star_id
04. proyecto_origen
05. region_origen
06. region_catalogo
07. subtipo_rrlyrae
08. subtipo_ogleiii
09. subtipo_ogleiv
10. subtipos_discordantes
11. transicion
12. grupo_analisis
13. dos_observaciones
14. periodo_catalogo
15. periodo_secundario
16. n_obs
17. n_obs_validas
18. ra
19. dec
20. separacion_fases_arcsec
21. archivo_origen
22. ruta_curva
23. archivo_periodos
24. archivo_ident
25. tipo_catalogo
26. seleccion_inicial
27. total_filas
28. formato_invalido
29. valores_no_numericos
30. error_no_positivo
31. subtipo_ident
32. subtipo_coincidente


In [19]:
assert len(catalogo_maestro) == 104_008
assert catalogo_maestro["observacion_id"].is_unique
assert catalogo_maestro["star_id_base"].nunique() == 70_269
assert (catalogo_maestro["n_obs_validas"] >= 300).all()
assert (catalogo_maestro["periodo_catalogo"] > 0).all()
assert catalogo_maestro["subtipo_coincidente"].all()
assert catalogo_maestro["seleccion_inicial"].all()

ruta_catalogo_maestro = Path(
    "../data/interim/catalogo_maestro.csv"
)

ruta_catalogo_maestro.parent.mkdir(
    parents=True,
    exist_ok=True,
)

catalogo_maestro.to_csv(
    ruta_catalogo_maestro,
    index=False,
    encoding="utf-8",
)

catalogo_verificacion = pd.read_csv(
    ruta_catalogo_maestro,
    encoding="utf-8",
)

print(f"Archivo exportado: {ruta_catalogo_maestro.resolve()}")
print(f"Filas guardadas: {len(catalogo_verificacion):,}")
print(f"Columnas guardadas: {len(catalogo_verificacion.columns)}")
print(
    "observacion_id únicos: "
    f"{catalogo_verificacion['observacion_id'].nunique():,}"
)
print(
    "star_id_base únicos: "
    f"{catalogo_verificacion['star_id_base'].nunique():,}"
)
print(
    "Tamaño: "
    f"{ruta_catalogo_maestro.stat().st_size / 1024**2:,.2f} MB"
)

C:\Users\adanm\AppData\Local\Temp\ipykernel_8512\1554390933.py:24: DtypeWarning: Columns (0: transicion) have mixed types. Specify dtype option on import or set low_memory=False.
  catalogo_verificacion = pd.read_csv(


Archivo exportado: C:\Users\adanm\Code\tesis\data\interim\catalogo_maestro.csv
Filas guardadas: 104,008
Columnas guardadas: 32
observacion_id únicos: 104,008
star_id_base únicos: 70,269
Tamaño: 34.91 MB
